# Аналитический отчёт о кино на основе данных MovieLens и IMDb


---

### Введение
В отчёте проведён анализ данных из коллекции MovieLens, дополненной информацией из IMDb. Цель — выявить основные тенденции релизов фильмов, распределение жанров, а также определить лидеров по популярности и качеству в каталоге фильмов.

In [ ]:
# %load movielens.py
#!/usr/bin/env python3
from collections import Counter, defaultdict
import sys
import datetime
import re
import os


class Ratings:
    """
    Analyzing data from ratings.csv
    """

    def __init__(self, path_to_the_file):

        self.ratings_path = path_to_the_file
        self.ratings_data = list(self.ratings_processor())
        self.movies = self.Movies(self.ratings_data)
        self.users = self.Users(self.ratings_data)

    def ratings_processor(self):

        with open(self.ratings_path, "rt", encoding="utf-8") as ratings_db:
            line = ratings_db.readline().split(",")
            while line:
                line = ratings_db.readline()
                if not line:
                    line = ratings_db.readline()
                    continue
                yield line.strip().split(",")

    class Movies:

        def __init__(self, ratings_data):

            self.ratings_data = ratings_data
            self.movies_dict = {
                key: value for key, value in self.movie_dictionary_reader()
            }

        def splitter(self, line):

            pattern = r',(?=(?:[^"]*"[^"]*")*[^"]*$)'
            parts = re.split(pattern, line)
            parts = [p.strip().strip('"') for p in parts]
            if len(parts) >= 3:
                return [parts[0], parts[1], parts[2]]
            else:
                return None

        def movie_dictionary_reader(self):
            with open("movies.csv", "rt", encoding="utf-8") as movies_db:
                line = movies_db.readline()
                while line:
                    line = movies_db.readline()
                    if not line:
                        line = movies_db.readline()
                        continue
                    parts = self.splitter(line)
                    if parts is None:
                        continue
                    yield int(parts[0]), parts[1]

        def dist_by_year(self):

            years_ratings_list = sorted(
                [
                    (datetime.datetime.fromtimestamp(float(row[3])).year)
                    for row in self.ratings_data
                    if len(row) > 3
                ]
            )
            ratings_by_year = dict(
                sorted(Counter(years_ratings_list).items(), key=lambda x: x[0])
            )
            return ratings_by_year

        def dist_by_rating(self):

            ratings_ratings_list = sorted(
                [round(float(row[2]), 2) for row in self.ratings_data if len(row) > 3]
            )
            ratings_distribution = dict(Counter(ratings_ratings_list))
            return ratings_distribution

        def top_by_num_of_ratings(self, n):

            mov_rating_list = [int(row[1]) for row in self.ratings_data if len(row) > 3]
            mov_rating_list = [
                self.movies_dict.get(movie_id, movie_id) for movie_id in mov_rating_list
            ]
            top_movies = sorted(
                Counter(mov_rating_list).most_common(n),
                key=lambda x: x[1],
                reverse=True,
            )

            return dict(top_movies)

        def median(self, lst):
            sorted_lst = sorted(lst)
            length = len(sorted_lst)
            mid = length // 2
            if length % 2 != 0:
                return sorted_lst[mid]
            else:
                return sorted_lst[mid - 1] / 2 + sorted_lst[mid] / 2

        def average(self, lst):

            length = len(lst)
            su = sum(lst)
            av = su / length
            return round(av, 2)

        def varience(self, lst):
            length = len(lst)
            if length < 2:
                return 0.0
            avg = sum(lst) / length
            var_sum = sum((x - avg) ** 2 for x in lst)
            var = var_sum / (length - 1)
            return round(var, 2)

        def top_by_ratings(self, n, metric="median"):

            mov_rating_list = [
                (int(row[1]), row[2]) for row in self.ratings_data if len(row) > 3
            ]
            mov_rating_list = [
                (self.movies_dict.get(movie_id, movie_id), round(float(rating), 2))
                for movie_id, rating in mov_rating_list
            ]
            ratings_by_movie = {}
            for movie, rating in mov_rating_list:
                ratings_by_movie.setdefault(movie, []).append(rating)
            if metric == "average":
                av_rating = [
                    (str(movie), self.average(ratings))
                    for movie, ratings in ratings_by_movie.items()
                ]
                top_movies_full = sorted(av_rating, key=lambda x: x[1], reverse=True)
                top_movies = top_movies_full[:n]
            elif metric == "median":
                median_rating = [
                    (str(movie), self.median(ratings))
                    for movie, ratings in ratings_by_movie.items()
                ]
                top_movies_full = sorted(
                    median_rating, key=lambda x: x[1], reverse=True
                )
                top_movies = top_movies_full[:n]
            else:
                raise Exception("Wrong_statistic")
            return dict(top_movies)

        def top_controversial(self, n):

            mov_rating_list = [
                (int(row[1]), row[2]) for row in self.ratings_data if len(row) > 3
            ]
            mov_rating_list = [
                (self.movies_dict.get(movie_id, movie_id), round(float(rating), 2))
                for movie_id, rating in mov_rating_list
            ]
            ratings_by_movie = {}
            for movie, rating in mov_rating_list:
                ratings_by_movie.setdefault(movie, []).append(rating)
            movies_varience = [
                (str(movie), self.varience(ratings))
                for movie, ratings in ratings_by_movie.items()
            ]
            top_movies_full = sorted(movies_varience, key=lambda x: x[1], reverse=True)
            top_movies = dict(top_movies_full[:n])
            return top_movies

    class Users(Movies):

        def users_by_ratings_num(self, n):

            users_dist = [(row[0]) for row in self.ratings_data if len(row) > 3]
            users = dict(Counter(users_dist).most_common(n))
            return users

        def users_by_avg_median_ratings(self, n, metric="median"):

            users_dist = sorted(
                [(row[0], float(row[2])) for row in self.ratings_data if len(row) > 3],
                key=lambda x: x[1],
            )
            ratings_by_user = {}
            for user, rating in users_dist:
                ratings_by_user.setdefault(user, []).append(rating)
            if metric == "average":
                av_rating = [
                    (user, round(self.average(ratings), 2))
                    for user, ratings in ratings_by_user.items()
                ]
                users_full = sorted(av_rating, key=lambda x: x[1], reverse=True)
                users = dict(users_full[:n])
            elif metric == "median":
                median_rating = [
                    (user, round(self.median(ratings), 2))
                    for user, ratings in ratings_by_user.items()
                ]
                users_full = sorted(median_rating, key=lambda x: x[1], reverse=True)
                users = dict(users_full[:n])
            else:
                raise Exception("Wrong_statistic")

            return users

        def users_by_varience(self, n):

            users_dist = [
                (row[0], float(row[2])) for row in self.ratings_data if len(row) > 3
            ]
            ratings_by_user = {}
            for user, rating in users_dist:
                ratings_by_user.setdefault(user, []).append(rating)
            users_varience = sorted(
                [
                    (user, round(self.varience(ratings), 2))
                    for user, ratings in ratings_by_user.items()
                ],
                key=lambda x: x[1],
                reverse=True,
            )
            users = users_varience[:n]

            return dict(users)


class Movies:
    """
    Analyzing data from movies.csv
    """

    def __init__(self, path_to_the_file):

        self.path_to_the_file = path_to_the_file
        self.movie_db = list(self.movie_reader())

    def movie_reader(self):

        with open(self.path_to_the_file, "rt", encoding="utf-8") as movies_db:
            line = movies_db.readline()
            while line:
                line = movies_db.readline()
                if not line:
                    line = movies_db.readline()
                    continue
                res = self.splitter(line)
                if res is None:
                    continue
                yield res

    def splitter(self, line):

        pattern = r"^(\d+),(.+\(\d{4}\)),(.+)$"
        match = re.match(pattern, line)
        if match:
            return [match.group(1), match.group(2), match.group(3)]
        else:
            return None

    def dist_by_release(self):

        years = [
            int(
                re.search(r"\((\d{4})\)", row[1]).group(1)
                if re.search(r"\((\d{4})\)", row[1])
                else "0"
            )
            for row in self.movie_db
        ]
        years = sorted(years, reverse=True)
        release_years = dict(Counter(years))
        return release_years

    def dist_by_genres(self):

        gen = [row[2] for row in self.movie_db if len(row) > 2]
        expanded_rows = []
        for row in gen:
            genres = row.split("|")
            for genre in genres:
                expanded_rows.append(genre)
        self.genres_counter = dict(Counter(expanded_rows).most_common())
        return self.genres_counter

    def most_genres(self, n):

        tit_gen = [(row[1], row[2]) for row in self.movie_db if len(row) > 2]
        expanded_rows = []
        for row in tit_gen:
            genres = row[1].split("|")
            for genre in genres:
                expanded_rows.append(row[0])
        movies = Counter(expanded_rows).most_common(n)
        return dict(movies)

    def genre_percentage_over_years(self, target_genre="Drama"):
        counts = defaultdict(int)
        total_per_year = defaultdict(int)

        for row in self.movie_db:
            if len(row) > 2:
                year_match = re.search(r"\((\d{4})\)", row[1])
                year = int(year_match.group(1)) if year_match else 0

                total_per_year[year] += 1

                genres = row[2].split("|")
                if target_genre in genres:
                    counts[year] += 1
        result = []
        for year in sorted(total_per_year):
            total = total_per_year[year]
            genre_count = counts.get(year, 0)
            percent = (genre_count / total * 100) if total > 0 else 0
            result.append((year, round(percent, 2)))

        return dict(result)


class Tags:

    def __init__(self, path_to_the_file):

        self.tag_path = path_to_the_file
        self.data = list(self.tag_processor())

    def tag_processor(self):

        with open(self.tag_path, "rt", encoding="utf-8") as tags_db:
            line = tags_db.readline()
            while line:
                line = tags_db.readline()
                yield line.strip().split(",")

    def most_words(self, n):

        tags_data = [row[2] for row in self.data if len(row) > 3]
        word_count = {tag: len(tag.split(" ")) for tag in tags_data}
        big_tags = sorted(
            Counter(word_count).most_common(n), key=lambda x: len(x[0]), reverse=True
        )
        return dict(big_tags)

    def longest(self, n):

        tags_data = {str(row[2]): len(row[2]) for row in self.data if len(row) > 3}
        big_tags = [
            x[0] for x in sorted(tags_data.items(), key=lambda x: x[1], reverse=True)
        ][:n]
        return big_tags

    def most_words_and_longest(self, n):

        list_most_word = set(self.most_words(n).keys())
        list_longest = set(self.longest(n))
        big_tags = sorted(
            list(list_most_word & list_longest), key=lambda x: len(x), reverse=True
        )

        return big_tags

    def most_popular(self, n):

        tags_data = [row[2] for row in self.data if len(row) > 3]
        popular_tags = dict(Counter(tags_data).most_common(n))
        return popular_tags

    def tags_with(self, word):

        tags_with_word = sorted(
            set([row[2] for row in self.data if len(row) > 3 and word in row[2]])
        )
        return tags_with_word


import requests
from bs4 import BeautifulSoup as bs


class Links:
    """
    Analyzing data from links.csv
    """

    def __init__(self, path_to_the_file):
        try:
            print("Links class initialization...")
            self.file_reader(path_to_the_file)
            self.dataset_path = "ex_links.tsv"
            if not os.path.isfile(self.dataset_path):
                self.data_exists = False
                self.error = "Вы не скачали наш датасет, скачайте или создайте свой"
                raise Exception(self.error)
            else:
                self.data_exists = True
                self.dict_from_data()
        except Exception as e:
            print(e)
        # [print(k, v) for k, v in self.ex_links_dict.items()]

    def file_reader(self, path_to_the_file):
        if not os.path.isfile(path_to_the_file):
            raise Exception("добавьте файл links.csv")
        else:
            self.links_dict = {}
            file = open(path_to_the_file, "r")
            header = file.readline()
            self.links_dict = {
                line.split(",")[0]: [elem.strip() for elem in line.split(",")[1:]]
                for line in file
            }
            file.close()
            return self.links_dict

    def dict_from_data(self):
        if not self.data_exists:
            print(self.error)
        else:
            print(f"'{self.dataset_path}' initialization...")
            ex_links_dict = {}
            with open(self.dataset_path, "r", encoding="utf-8") as data:
                header_skip = data.readline()
                for line in data:
                    splitted = line.split("\t")
                    ex_links_dict.update({splitted[0]: splitted[1:]})
            self.ex_links_dict = ex_links_dict
            print("successfully initialized\nNow cleaning...")
            if len(header_skip.split("\t")) > 9:
                for k, v in self.ex_links_dict.items():
                    v[4] = self.Cleaner.clean_budget(v[4])
                    v[5] = self.Cleaner.clean_budget(v[5])
                    v[3] = self.Cleaner.clean_time(v[3])
                self.flag = True
            else:
                self.flag = False
                self.error = "Столбцов в датасете не верное кол-во."
            print("You can call all methods now")

    def get_imdb(self, list_of_movies, list_of_fields):
        import time

        def parse(imdb_id, list_of_fields):
            headers = {
                "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
                "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36",
            }
            all_fields = {
                "title": "h1[data-testid='hero__pageTitle'], h1[data-testid='hero-title-block__title']",
                "director": "li.ipc-metadata-list__item:has(span:-soup-contains('Director')) a",
                "release_date": "li[data-testid='title-details-releasedate'] a",
                "runtime": "li[data-testid='title-techspec_runtime'] div ul li",
                "budget": "li[data-testid='title-boxoffice-budget'] > div > ul > li > span:last-child",
                "cumulative_gross": "li[data-testid='title-boxoffice-cumulativeworldwidegross'] > div > ul > li > span:last-child",
                "production_companies": "li[data-testid='title-details-companies'] li a",
                "countries_of_origin": "li[data-testid='title-details-origin'] li a",
                "languages": "li[data-testid='title-details-languages'] li a",
            }
            fields = {
                field: selector
                for field, selector in all_fields.items()
                if field in list_of_fields
            }
            if not fields:
                raise ValueError("No valid fields to parse.")
            for attempt in range(5):
                try:
                    html = requests.get(
                        f"https://www.imdb.com/title/tt{imdb_id}/", headers=headers
                    )
                    time.sleep(0.01)
                    html.raise_for_status()
                    soup = bs(html.text, "html.parser")

                    data = {}
                    for key, selector in fields.items():
                        items = soup.select(selector)
                        data[key] = list(
                            set(
                                [i.get_text(strip=True) for i in items]
                                if items
                                else ["No Data"]
                            )
                        )
                    return data

                except Exception as e:
                    print(f"\n[{imdb_id}] Ошибка при парсинге ({attempt + 1}/{5}): {e}")
                    time.sleep(7)
                    if attempt == 4:
                        return {field: ["No Data"] for field in fields}

        imdb_info = []
        list_of_movies = [
            str(elem)
            for elem in sorted(list_of_movies, reverse=True)
            if str(elem) in self.links_dict.keys()
        ]
        for movie_id in list_of_movies:
            imdb_id = self.links_dict[movie_id][0]
            imdb_info.extend(
                [[movie_id]] + [v for v in parse(imdb_id, list_of_fields).values()]
            )
        # [print(*elem, end="\n\n") for elem in imdb_info]
        # print(f"function print \t {imdb_info}")
        # print(imdb_info)
        return imdb_info

        """
        The method returns a list of lists [movieId, field1, field2, field3, ...] for the list of movies given as the argument (movieId).
                For example, [movieId, Director, Budget, Cumulative Worldwide Gross, Runtime].
                The values should be parsed from the IMDB webpages of the movies.
             Sort it by movieId descendingly.
        """

    def top_directors(self, n):
        if not self.data_exists or not self.flag:
            print(self.error)
        else:
            counts = Counter(v[1] for v in self.ex_links_dict.values())
            temp_sort = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
            sorted_counts = {e[0]: e[1] for e in temp_sort if e[0] != "No Data"}
            """
            The method returns a dict with top-n directors where the keys are directors and
            the values are numbers of movies created by them. Sort it by numbers descendingly.
            """
            if n > 0:
                directors = dict(list(sorted_counts.items())[:n])
            else:
                raise ValueError("n must be > 0")
            return directors

    def most_expensive(self, n):
        if not self.data_exists or not self.flag:
            print(self.error)
        else:
            d = self.ex_links_dict
            sorted_d = sorted(
                d.items(), key=lambda item: ((item[1][4] or 0), item[0]), reverse=True
            )
            budgets = {v[0]: f"${v[4]}" for k, v in sorted_d if v[4] is not None}

            """
            The method returns a dict with top-n movies where the keys are movie titles and
            the values are their budgets. Sort it by budgets descendingly.
            """
            if n > 0:
                budgets = dict(list(budgets.items())[:n])
            return budgets

    def most_profitable(self, n):
        if not self.data_exists or not self.flag:
            print(self.error)
        else:
            d = self.ex_links_dict

            profits = {}
            for title, v in d.items():
                if len(v) > 5 and v[4] is not None and v[5] is not None:
                    profits[v[0]] = v[5] - v[4]

            sorted_profits = dict(
                sorted(profits.items(), key=lambda x: x[1], reverse=True)
            )
            budgets = {k: f"${v}" for k, v in sorted_profits.items()}
            """
            The method returns a dict with top-n movies where the keys are movie titles and
            the values are the difference between cumulative worldwide gross and budget.
            Sort it by the difference descendingly.
            """
            return dict(list(budgets.items())[:n])

    def longest(self, n):
        if not self.data_exists or not self.flag:
            print(self.error)
        else:
            d = self.ex_links_dict
            sorted_d = sorted(
                d.items(), key=lambda item: ((item[1][3] or 0), item[0]), reverse=True
            )
            runtimes = {v[0]: f"{v[3]} mins" for k, v in sorted_d if v[3] is not None}

            """
            The method returns a dict with top-n movies where the keys are movie titles and
            the values are their runtime. If there are more than one version – choose any.
            Sort it by runtime descendingly.
            """
            if n > 0:
                runtimes = dict(list(runtimes.items())[:n])

            return runtimes

    def top_cost_per_minute(self, n):
        if not self.data_exists or not self.flag:
            print(self.error)
        else:
            d = self.ex_links_dict

            costs = {}
            for title, v in d.items():
                if len(v) > 5 and v[3] not in (None, 0) and v[4] is not None:
                    costs[v[0]] = round(v[4] / v[3], 2)

            sorted_costs = dict(sorted(costs.items(), key=lambda x: x[1], reverse=True))
            """
                    The method returns a dict with top-n movies where the keys are movie titles and
            the values are the budgets divided by their runtime. The budgets can be in different currencies – do not pay attention to it.
                The values should be rounded to 2 decimals. Sort it by the division descendingly.
            """
            return dict(list(sorted_costs.items())[:n])

    def create_dataset(self, list_of_movies, list_of_fields):
        print("Введите название вашего датасета:")
        self.dataset_path = f"{input()}.tsv"
        list_of_fields = [
            field.lower()
            for field in list_of_fields
            if field
            in [
                "title",
                "director",
                "release_date",
                "runtime",
                "budget",
                "cumulative_gross",
                "production_companies",
                "countries_of_origin",
                "languages",
            ]
        ]
        [
            print(f"movieId {movie_id} not in links.csv")
            for movie_id in list_of_movies
            if str(movie_id) not in self.links_dict.keys()
        ]
        with open(self.dataset_path, "w", encoding="utf-8") as out:
            tab = "\t"
            out.write(f"movieId{tab}{tab.join(list_of_fields)}\n")
            for key in sorted(list_of_movies, reverse=True):
                result = self.get_imdb([key], list_of_fields)

                s_result = "\t".join(
                    ",".join(str(item).strip() for item in sublist)
                    for sublist in result
                )
                if s_result:
                    out.write(s_result + "\n")

                print(key, end=", ")
            self.data_exists = True
            print(f"\nDataset '{self.dataset_path}' created successfull")

    class Cleaner:
        @staticmethod
        def clean_budget(value):

            if not value or value == "No Data":
                return None
            if not isinstance(value, str):
                return value
            if (
                "$" not in value
                and "usd" not in value.lower()
                and "dollar" not in value.lower()
            ):
                return None

            match = re.search(r"[\d,]+(?:\.\d+)?", value)
            if not match:
                return None

            number_str = match.group(0).replace(",", "")
            try:
                return int(float(number_str))
            except ValueError:
                return None

        @staticmethod
        def clean_time(value):

            if not value or not isinstance(value, str):
                return None
            match = re.search(r"\((\d+)\s*min\)", value)
            if match:
                return int(match.group(1))
            h = m = 0
            if "h" in value:
                h_match = re.search(r"(\d+)\s*h", value)
                if h_match:
                    h = int(h_match.group(1))
            if "m" in value:
                m_match = re.search(r"(\d+)\s*m", value)
                if m_match:
                    m = int(m_match.group(1))
            total = h * 60 + m
            return total if total > 0 else None


class Tests:

    def test_dist_by_year(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.movies.dist_by_year()
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, int)
            assert isinstance(v, int)
        keys = list(result.keys())
        assert keys == sorted(keys)

    def test_dist_by_rating(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.movies.dist_by_rating()
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, float)
            assert isinstance(v, int)
        keys = list(result.keys())
        assert keys == sorted(keys)

    def test_top_by_num_of_ratings(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.movies.top_by_num_of_ratings(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, int)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_top_by_ratings(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.movies.top_by_ratings(10, "average")
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, float)
        val = list(result.values())
        assert val == sorted(val, reverse=False)
        result = r_obj.movies.top_by_ratings(10, "median")
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, float)
        val = list(result.values())

    def test_top_controversial(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.movies.top_controversial(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, float)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_users_by_ratings_num(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.users.users_by_ratings_num(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, int)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_users_by_avg_median_ratings(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.users.users_by_avg_median_ratings(10, "average")
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, float)
        val = list(result.values())
        assert val == sorted(val, reverse=True)
        result = r_obj.users.users_by_avg_median_ratings(10, "median")
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, float)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_users_by_varience(self):
        r_obj = Ratings("ratings.csv")
        result = r_obj.users.users_by_varience(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, float)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_dist_by_release(self):
        m_obj = Movies("movies.csv")
        result = m_obj.dist_by_release()
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, int)
            assert isinstance(v, int)
        val = list(result.keys())
        assert val == sorted(val, reverse=True)

    def test_dist_by_genres(self):
        m_obj = Movies("movies.csv")
        result = m_obj.dist_by_genres()
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, int)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_most_genres(self):
        m_obj = Movies("movies.csv")
        result = m_obj.most_genres(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, int)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_genre_percentage_over_years(self):
        m_obj = Movies("movies.csv")
        result = m_obj.genre_percentage_over_years("Drama")
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, int)
            assert isinstance(v, float)
        val = list(result.keys())
        assert val == sorted(val, reverse=False)

    def test_most_words(self):
        t_obj = Tags("tags.csv")
        result = t_obj.most_words(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, int)
        val = list(result.keys())
        assert val == sorted(val, key=lambda x: len(x), reverse=True)

    def test_longest(self):
        t_obj = Tags("tags.csv")
        result = t_obj.longest(10)
        assert isinstance(result, list)
        for k in result:
            assert isinstance(k, str)
        val = result
        assert val == sorted(val, key=lambda x: len(x), reverse=True)

    def test_most_words_and_longest(self):
        t_obj = Tags("tags.csv")
        result = t_obj.most_words_and_longest(10)
        assert isinstance(result, list)
        for k in result:
            assert isinstance(k, str)
        val = result
        assert val == sorted(val, key=lambda x: len(x), reverse=True)

    def test_most_popular(self):
        t_obj = Tags("tags.csv")
        result = t_obj.most_popular(10)
        assert isinstance(result, dict)
        for k, v in result.items():
            assert isinstance(k, str)
            assert isinstance(v, int)
        val = list(result.values())
        assert val == sorted(val, reverse=True)

    def test_tags_with(self):
        t_obj = Tags("tags.csv")
        result = t_obj.tags_with("Disney")
        assert isinstance(result, list)
        for k in result:
            assert isinstance(k, str)
        val = result
        assert "Disney" in val


if __name__ == "__main__":
    # import sys

    # try:
    #     m_path = "movies.csv"
    #     r_path = "ratings.csv"
    #     t_path = "tags.csv"
    #     obj_r = Ratings(r_path)
    #     obj_m = Movies(m_path)
    #     obj_t = Tags(t_path)
    #     print(obj_m.genre_percentage_over_years())
    #     print()
    #     print(obj_r.movies.top_by_ratings(20, "average"))
    #     print(obj_t.most_popular(10))

    # except Exception as e:
    #     print(e)


{1903: 0.0, 1908: 0.0, 1916: 50.0, 1919: 100.0, 1920: 0.0, 1923: 0.0, 1924: 66.67, 1925: 100.0, 1926: 100.0, 1927: 66.67, 1928: 0.0, 1930: 66.67, 1931: 63.64, 1932: 57.14, 1933: 40.0, 1934: 28.57, 1935: 37.5, 1936: 37.5, 1937: 81.82, 1938: 58.33, 1939: 62.5, 1940: 57.14, 1941: 50.0, 1942: 62.5, 1943: 71.43, 1944: 75.0, 1945: 22.22, 1946: 64.29, 1947: 54.55, 1948: 64.29, 1949: 64.71, 1950: 47.37, 1951: 50.0, 1952: 45.45, 1953: 56.52, 1954: 53.33, 1955: 57.69, 1956: 58.82, 1957: 45.0, 1958: 33.33, 1959: 42.86, 1960: 50.0, 1961: 43.48, 1962: 61.29, 1963: 42.86, 1964: 38.46, 1965: 43.33, 1966: 41.38, 1967: 53.33, 1968: 46.88, 1969: 50.0, 1970: 60.0, 1971: 45.45, 1972: 58.33, 1973: 53.66, 1974: 40.74, 1975: 46.43, 1976: 43.33, 1977: 42.22, 1978: 31.25, 1979: 46.15, 1980: 38.1, 1981: 33.85, 1982: 29.03, 1983: 30.0, 1984: 27.54, 1985: 32.38, 1986: 35.4, 1987: 36.89, 1988: 38.35, 1989: 44.74, 1990: 39.67, 1991: 41.13, 1992: 41.3, 1993: 46.88, 1994: 46.77, 1995: 46.6, 1996: 44.29, 1997: 48.6, 1

In [1]:
from movielens import Links, Ratings, Movies, Tags

In [2]:
obj_m = Movies('movies.csv')
obj_r = Ratings('ratings.csv')
obj_t = Tags('tags.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'movies.csv'

#### Распределение кинофильмов по году их производства

In [12]:
result = obj_m.dist_by_release()
print(result)
print()
%timeit obj_m.dist_by_release()

{2018: 39, 2017: 139, 2016: 216, 2015: 267, 2014: 246, 2013: 190, 2012: 176, 2011: 202, 2010: 201, 2009: 216, 2008: 208, 2007: 222, 2006: 227, 2005: 210, 2004: 224, 2003: 220, 2002: 257, 2001: 235, 2000: 218, 1999: 199, 1998: 194, 1997: 214, 1996: 219, 1995: 206, 1994: 186, 1993: 160, 1992: 138, 1991: 124, 1990: 121, 1989: 114, 1988: 133, 1987: 122, 1986: 113, 1985: 105, 1984: 69, 1983: 70, 1982: 62, 1981: 65, 1980: 63, 1979: 52, 1978: 48, 1977: 45, 1976: 30, 1975: 28, 1974: 27, 1973: 41, 1972: 24, 1971: 33, 1970: 20, 1969: 24, 1968: 32, 1967: 30, 1966: 29, 1965: 30, 1964: 26, 1963: 21, 1962: 31, 1961: 23, 1960: 22, 1959: 21, 1958: 21, 1957: 20, 1956: 17, 1955: 26, 1954: 15, 1953: 23, 1952: 11, 1951: 10, 1950: 19, 1949: 17, 1948: 14, 1947: 11, 1946: 14, 1945: 9, 1944: 12, 1943: 7, 1942: 16, 1941: 14, 1940: 14, 1939: 16, 1938: 12, 1937: 11, 1936: 16, 1935: 8, 1934: 7, 1933: 10, 1932: 7, 1931: 11, 1930: 3, 1928: 1, 1927: 3, 1926: 3, 1925: 2, 1924: 3, 1923: 2, 1920: 1, 1919: 1, 1916: 2, 1

Мы заметили, что количество фильмов, выпускамых в год, чётко коррелирует с ускорением темпа нашей жизни


**Таким образом можно выделить несколько периодов:**

- Заря кинематографа 1984 - 1899 г.г - в год снималось не более 10 кинофильмов.
- Период становления кино 1900 - 1926 г.г - в год снималось от 10 до 50 кинофильмов.
- Период поступательного развития отрасли 1926 - 1990 г.г - кино, кроме развлекательной, начинает выполнять еще и культурно - идеологическую роль. Количество фильмов плавно растет со скромных 50 до почти 500 фильмов в год. 
- Первый скачок скорости производства 1990 - 2000 г.г - по мере того, за 10 лет количество фильмов почти достигает 1000 штук в год. Именно этому периоду мы обязаны появлены появлением большинства наиболее успешнх и значимых названий кинофильмов, которые до сих пор привлекают зрителей.
- Второй скачок, 2000 - н.в - количество фильмов, выпускаемых в год продолжает расти, но темп роста снизился, хотя в  абсолютных числах, сейчас за год выпускается больше кино, чем за десять лет в 1970-е



---

#### Распределение кинофильмов по жанрам

In [13]:
result = obj_m.dist_by_genres()
print(result)
print()
%timeit obj_m.dist_by_genres()

{'Drama': 3304, 'Comedy': 3044, 'Action': 1492, 'Thriller': 1440, 'Romance': 1250, 'Adventure': 987, 'Crime': 946, 'Sci-Fi': 811, 'Horror': 748, 'Fantasy': 598, 'Children': 503, 'Animation': 499, 'Mystery': 433, 'Documentary': 355, 'War': 289, 'Musical': 252, 'IMAX': 132, 'Western': 128, 'Film-Noir': 65, '(no genres listed)': 25}

3.56 ms ± 118 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- На основе анализа данного графика можно сделать вывод, что наибольшее количество фильмов снято в жанре **"Драма"**
- Второе место занимает жанр **"Комедия"**
- Третье место практически поровну поделили **"Триллеры"**, **"Боевики"** и **"Романтическое кино"**. При этом первые две категории содержать в себе практически столько же кинофильмов, как все остальные вместе взятые. Впрочем, необходимо помнить, что в последние пару десятилетий уже не снимают моножанровых фильмов, поэтому данный анализ не вполне корректен

#### Процентное соотношение фильмов в жанре Драма по годам

In [14]:
result = obj_m.genre_percentage_over_years('Drama')
print(result)
print()
%timeit obj_m.genre_percentage_over_years('Drama')

{1903: 0.0, 1908: 0.0, 1916: 50.0, 1919: 100.0, 1920: 0.0, 1923: 0.0, 1924: 66.67, 1925: 100.0, 1926: 100.0, 1927: 66.67, 1928: 0.0, 1930: 66.67, 1931: 63.64, 1932: 57.14, 1933: 40.0, 1934: 28.57, 1935: 37.5, 1936: 37.5, 1937: 81.82, 1938: 58.33, 1939: 62.5, 1940: 57.14, 1941: 50.0, 1942: 62.5, 1943: 71.43, 1944: 75.0, 1945: 22.22, 1946: 64.29, 1947: 54.55, 1948: 64.29, 1949: 64.71, 1950: 47.37, 1951: 50.0, 1952: 45.45, 1953: 56.52, 1954: 53.33, 1955: 57.69, 1956: 58.82, 1957: 45.0, 1958: 33.33, 1959: 42.86, 1960: 50.0, 1961: 43.48, 1962: 61.29, 1963: 42.86, 1964: 38.46, 1965: 43.33, 1966: 41.38, 1967: 53.33, 1968: 46.88, 1969: 50.0, 1970: 60.0, 1971: 45.45, 1972: 58.33, 1973: 53.66, 1974: 40.74, 1975: 46.43, 1976: 43.33, 1977: 42.22, 1978: 31.25, 1979: 46.15, 1980: 38.1, 1981: 33.85, 1982: 29.03, 1983: 30.0, 1984: 27.54, 1985: 32.38, 1986: 35.4, 1987: 36.89, 1988: 38.35, 1989: 44.74, 1990: 39.67, 1991: 41.13, 1992: 41.3, 1993: 46.88, 1994: 46.77, 1995: 46.6, 1996: 44.29, 1997: 48.6, 1

#### Процентное соотношение фильмов в жанре Комедия по годам

In [15]:
result = obj_m.genre_percentage_over_years('Comedy')
print(result)
print()
%timeit obj_m.genre_percentage_over_years('Comedy')

{1903: 0.0, 1908: 100.0, 1916: 0.0, 1919: 100.0, 1920: 100.0, 1923: 100.0, 1924: 33.33, 1925: 0.0, 1926: 0.0, 1927: 33.33, 1928: 100.0, 1930: 33.33, 1931: 27.27, 1932: 42.86, 1933: 50.0, 1934: 42.86, 1935: 62.5, 1936: 43.75, 1937: 9.09, 1938: 66.67, 1939: 43.75, 1940: 42.86, 1941: 50.0, 1942: 31.25, 1943: 0.0, 1944: 16.67, 1945: 11.11, 1946: 0.0, 1947: 45.45, 1948: 21.43, 1949: 35.29, 1950: 31.58, 1951: 20.0, 1952: 54.55, 1953: 30.43, 1954: 20.0, 1955: 15.38, 1956: 11.76, 1957: 30.0, 1958: 42.86, 1959: 19.05, 1960: 18.18, 1961: 30.43, 1962: 16.13, 1963: 33.33, 1964: 38.46, 1965: 40.0, 1966: 44.83, 1967: 26.67, 1968: 21.88, 1969: 29.17, 1970: 30.0, 1971: 33.33, 1972: 37.5, 1973: 24.39, 1974: 29.63, 1975: 28.57, 1976: 36.67, 1977: 40.0, 1978: 31.25, 1979: 36.54, 1980: 30.16, 1981: 33.85, 1982: 40.32, 1983: 37.14, 1984: 47.83, 1985: 41.9, 1986: 41.59, 1987: 54.1, 1988: 48.87, 1989: 48.25, 1990: 43.8, 1991: 46.77, 1992: 44.93, 1993: 41.25, 1994: 47.85, 1995: 35.44, 1996: 41.1, 1997: 35.51,

### Анализ меток

#### Топ-10 тэгов по количеству слов в тэге

In [16]:
result = obj_t.most_words(10)
print(result)
print()
%timeit obj_t.most_words(10)

{'Something for everyone in this one... saw it without and plan on seeing it with kids!': 16, 'the catholic church is the most corrupt organization in history': 10, 'villain nonexistent or not needed for good story': 8, '06 Oscar Nominated Best Movie - Animation': 7, 'stop using useless characters for filler': 6, 'Oscar (Best Effects - Visual Effects)': 6, 'It was melodramatic and kind of dumb': 7, 'Oscar (Best Music - Original Score)': 6, 'Everything you want is here': 5, 'based on a true story': 5}

902 μs ± 34.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


---

Таким образом, рейтинг тегов по длине помогает понять, как именно аудитория выражает мнения о фильмах — через краткие яркие ярлыки или через более длинные, раскрывающие контекст фразы. Это может быть полезно для анализа предпочтений зрителей и улучшения систем рекомендаций или фильтрации контента

**На основе приведённых данных можно сделать следующие выводы:**

**Тема критики и недовольства** аудитории заметно прослеживается, например, по следующим тегам:
- "the catholic church is the most corrupt organization in history"
- "2 villain nonexistent or not needed for good story"
- "stop using useless characters for filler"
- "It was melodramatic and kind of dumb"
Такая картина может указывать на частые жалобы на сюжетные и тематические аспекты фильмов.

**Признание и награды** также популярны: теги с указанием на номинации и премии «Оскар» показывают интерес пользователей к качеству и признанию фильмов:
- "06 Oscar Nominated Best Movie - Animation"
- "Oscar (Best Effects - Visual Effects)"
- "Oscar (Best Music - Original Score)"
 
Тег **"Everything you want is here"** говорит о том, что некоторые фильмы воспринимаются как полноформатные и удовлетворяющие разнообразные ожидания зрителей.
Частота упоминания темы **"based on a true story"** говорит об интересе к фильмам, основанным на реальных событиях.

---

#### Топ-10 самых длинных тегов

In [17]:
result = obj_t.longest(10)
print(result)
print()
%timeit obj_t.longest(10)

['Something for everyone in this one... saw it without and plan on seeing it with kids!', 'the catholic church is the most corrupt organization in history', 'villain nonexistent or not needed for good story', 'r:disturbing violent content including rape', '06 Oscar Nominated Best Movie - Animation', 'stop using useless characters for filler', 'Academy award (Best Supporting Actress)', 'Oscar (Best Effects - Visual Effects)', 'audience intelligence underestimated', 'It was melodramatic and kind of dumb']

703 μs ± 16.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


---

#### Топ-10 самых длинных тегов с наибольшим количеством слов

In [18]:
result = obj_t.most_words_and_longest(10)
print(result)
print()
%timeit obj_t.most_words_and_longest(10)

['Something for everyone in this one... saw it without and plan on seeing it with kids!', 'the catholic church is the most corrupt organization in history', 'villain nonexistent or not needed for good story', '06 Oscar Nominated Best Movie - Animation', 'stop using useless characters for filler', 'Oscar (Best Effects - Visual Effects)', 'It was melodramatic and kind of dumb']

1.67 ms ± 39 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


#### Топ-10 тэгов по количеству запросов

In [19]:
result = obj_t.most_popular(10)
print(result)
print()
%timeit obj_t.most_popular(10)

{'In Netflix queue': 131, 'atmospheric': 36, 'superhero': 24, 'thought-provoking': 24, 'funny': 23, 'Disney': 23, 'surreal': 23, 'religion': 22, 'sci-fi': 21, 'dark comedy': 21}

556 μs ± 60.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


- Тег **«Netflix»** сам по себе показывает, что речь идет о контенте платформы, и он служит как обобщающий тег для всей выборки.
- Самый частый тег — **«atmospheric»**, что говорит о высоком спросе на фильмы и сериалы с насыщенной и выразительной атмосферой, которые способны погрузить зрителя в уникальный мир.
- Далее по популярности идут **«superhero»** и **«thought-provoking»** (по 24), что отражает интерес аудитории к динамичным экшенам с супергероями и одновременно к фильмам, вызывающим глубокие размышления.
- Теги **«funny»** (23), **«Disney»** (23) и **«surreal»** (23) близки по частоте, показывая, что юмор, семейный контент и необычный, сюрреалистический стиль ценятся почти одинаково.
- Теги **«religion»** (22), **«sci-fi»** (21) и **«dark comedy»** (21) немного менее популярны, но все равно в топе, что свидетельствует о наличии спроса на религиозные темы, научную фантастику и тёмный юмор.

#### Тэги Netflix

In [20]:
result = obj_t.tags_with('Netflix')
print(result)
print()
%timeit obj_t.tags_with('Netflix')

['In Netflix queue', 'No DVD at Netflix', 'Not available from Netflix']

309 μs ± 1.53 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


---

#### Распределение зрительской оценки фильма по годам

In [21]:
result = obj_r.movies.dist_by_year()
print(result)
print()
%timeit obj_r.movies.dist_by_year()

{1996: 6040, 1997: 1916, 1998: 507, 1999: 2439, 2000: 10061, 2001: 3922, 2002: 3478, 2003: 4014, 2004: 3279, 2005: 5813, 2006: 4059, 2007: 7114, 2008: 4351, 2009: 4158, 2010: 2300, 2011: 1690, 2012: 4657, 2013: 1664, 2014: 1439, 2015: 6616, 2016: 6702, 2017: 8199, 2018: 6418}

77.6 ms ± 984 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Распределение количества оценок

In [22]:
result = obj_r.movies.dist_by_rating()
print(result)
print()
%timeit obj_r.movies.dist_by_rating()

{0.5: 1370, 1.0: 2811, 1.5: 1791, 2.0: 7551, 2.5: 5550, 3.0: 20047, 3.5: 13136, 4.0: 26818, 4.5: 8551, 5.0: 13211}

53.8 ms ± 397 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Топ-10 фильмов по количеству оценок за все время

In [23]:
result = obj_r.movies.top_by_num_of_ratings(10)
print(result)
print()
%timeit obj_r.movies.top_by_num_of_ratings(10)

{'Forrest Gump (1994)': 329, 'Shawshank Redemption, The (1994)': 317, 'Pulp Fiction (1994)': 307, 'Silence of the Lambs, The (1991)': 279, 'Matrix, The (1999)': 278, 'Star Wars: Episode IV - A New Hope (1977)': 251, 'Jurassic Park (1993)': 238, 'Braveheart (1995)': 237, 'Terminator 2: Judgment Day (1991)': 224, "Schindler's List (1993)": 220}

23.3 ms ± 2.17 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


В списке представлены культовые и широко известные фильмы, такие как "Forrest Gump", "Shawshank Redemption", "Pulp Fiction" и "Star Wars", что говорит о их большой популярности и высокой вовлечённости аудитории.

Таким образом, можно сказать, что фильмы с большим количеством оценок пользуются устойчивым вниманием и заслуживают статуса классики или культовых произведений в киноиндустрии

#### Средний рейтинг топ-10 фильмов

In [24]:
result = obj_r.movies.top_by_ratings(10, 'average')
print(result)
print()
%timeit obj_r.movies.top_by_ratings(10, 'average')

{'The Jinx: The Life and Deaths of Robert Durst (2015)': 5.0, 'Galaxy of Terror (Quest) (1981)': 5.0, 'Alien Contamination (1980)': 5.0, "I'm the One That I Want (2000)": 5.0, 'Lesson Faust (1994)': 5.0, 'Assignment, The (1997)': 5.0, 'Mephisto (1981)': 5.0, 'Black Mirror': 5.0, 'Dylan Moran: Monster (2004)': 5.0, 'Bill Hicks: Revelations (1993)': 5.0}

77.8 ms ± 1.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Корреляция кол-ва зрителей и среднего рейтинга фильма

In [25]:
result = obj_r.users.users_by_avg_median_ratings(10, 'average')
print(result)
print()
%timeit obj_r.users.users_by_avg_median_ratings(10, 'average')

{'53': 5.0, '251': 4.87, '515': 4.85, '25': 4.81, '30': 4.74, '523': 4.69, '348': 4.67, '171': 4.63, '452': 4.56, '122': 4.55}

49.1 ms ± 1.77 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


В целом наблюдается тенденция, что фильмы с **большим количеством зрителей** имеют несколько **ниже средний рейтин**г, тогда как фильмы с **меньшей аудиторией** имеют чуть более **высокий средний рейтинг**.

Это может объясняться тем, что фильмы с широкой массовой аудиторией часто ориентированы на более широкий спектр вкусов, что приводит к более смешанным оценкам. В то же время менее популярные фильмы могут привлекать более узкую и заинтересованную аудиторию, которая ставит более высокие оценки.

#### Корреляция кол-ва оценок и кол-ва зрителей

In [26]:
result = obj_r.users.users_by_ratings_num(10)
print(result)
print()
%timeit obj_r.users.users_by_ratings_num(10)

{'414': 2698, '599': 2478, '474': 2108, '448': 1864, '274': 1346, '610': 1302, '68': 1260, '380': 1218, '606': 1115, '288': 1055}

10.8 ms ± 88.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Между количеством зрителей и количеством оценок есть прямая связь: фильмы с большим числом зрителей обычно получают больше оценок, так как больше людей участвуют в оценивании.

Такая вариативность указывает на разный уровень вовлеченности зрителей в процесс оценки фильмов, что важно учитывать при анализе рейтингов и популярности

---

### Анализ кинофильмов с IMDB

Для корректной работы этого класса необходимо скачать по ячейке уже готовый датасет

In [ ]:
!gdown -O ex_links.tsv 1UeuodwLYR5PJuPQluivrYj5N1gASBCiT

In [31]:
linker = Links("links.csv")
print(linker)
print()
%timeit Links("links.csv")

Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now

Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now
Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now
Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now
Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now
Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now
Links class initialization...
'ex_links.tsv' initialization...
successfully initialized
Now cleaning...
You can call all methods now
Links class initialization...
'ex_links.tsv' initialization...
succe

#### Информация о интересующих вас фильмах через IMDB  
Передайте как аргументы два списка! 
Список фильмов которые интересны и список полей которые хотите увидеть  
MovieId_list и Fields_list  
поддреживаемые поля :  
*"title", "director", "release_date", "runtime", "budget", "cumulative_gross", "production_companies", "countries_of_origin", "languages"* 

In [34]:
result = linker.get_imdb([1, 2, 14, 13], ["title", "director", "budget"])
print(result)
print()
%timeit linker.get_imdb([1, 2, 14, 13],["title", "director", "budget"])

[['14'], ['Никсон'], ['Oliver Stone'], ['$44,000,000 (estimated)'], ['13'], ['Балто'], ['Simon Wells'], ['No Data'], ['2'], ['Джуманджи'], ['Joe Johnston'], ['$65,000,000 (estimated)'], ['1'], ['История игрушек'], ['John Lasseter'], ['$30,000,000 (estimated)']]

5.98 s ± 506 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


#### Топ-10 режисеров по количеству фильмов

In [35]:
result = linker.top_directors(10)
print(result)
print()
%timeit linker.top_directors(10)

{'Woody Allen,Woody Allen,Woody Allen': 45, 'Alfred Hitchcock,Alfred Hitchcock,Alfred Hitchcock': 37, 'Steven Spielberg,Steven Spielberg,Steven Spielberg': 32, 'Clint Eastwood,Clint Eastwood,Clint Eastwood': 31, 'Martin Scorsese,Martin Scorsese,Martin Scorsese': 29, 'Ridley Scott,Ridley Scott,Ridley Scott': 24, 'Ron Howard,Ron Howard,Ron Howard': 24, 'Steven Soderbergh,Steven Soderbergh,Steven Soderbergh': 22, 'Barry Levinson,Barry Levinson,Barry Levinson': 21, 'Spike Lee,Spike Lee,Spike Lee': 21}

4.05 ms ± 117 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Среди режиссеров с наибольшим количеством фильмов лидирует **Вуди Аллен** с 48 картинами, за ним следуют **Альфред Хичкок** (37) и **Стивен Спилберг** (32). Эти режиссеры характеризуются высокой продуктивностью на протяжении длительного периода времени. 
Далее идут такие мастера, как **Клинт Иствуд** (31), **Мартин Скорсезе** (29) и **Ридли Скотт** (24), которые наряду с количеством фильмов обладают и значительным влиянием на индустрию.

#### Топ-10 фильмов по бюджету

In [36]:
result = linker.most_expensive(10)
print(result)
print()
%timeit linker.most_expensive(10)

{'Мстители: Война бесконечности': '$321000000', 'Звёздные войны: Последние джедаи': '$317000000', 'Пираты Карибского моря: На краю света': '$300000000', 'Лига справедливости': '$300000000', 'Хан Соло: Звездные войны. Истории': '$275000000', 'Возвращение Супермена': '$270000000', 'Рапунцель: Запутанная история': '$260000000', 'Человек-паук 3: Враг в отражении': '$258000000', 'Джон Картер': '$250000000', 'Тёмный рыцарь: Возрождение легенды': '$250000000'}

6.77 ms ± 384 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Главные инвестиции идут в крупные франшизы и проекты с широкой узнаваемостью. Это такие фильмы, как **«Мстители: Война бесконечности»**, **«Звёздные войны»** и **«Пираты Карибского моря»**.

Высокий бюджет обоснован ожиданиями массового зрительского интереса и большого кассового успеха. Такие фильмы часто базируются на популярных брендах, комиксах, литературе или предыдущих успешных частях.
Большие бюджеты направлены на фильмы с масштабными спецэффектами и визуальными эффектами, что требует серьезных затрат на технологии, съемку и постпродакшн
Важным фактором является участие звездных актеров и известных режиссеров, которые требуют высоких гонораров, что увеличивает общий бюджет.

#### Топ-10 фильмов по кассовым сборам

In [37]:
result = linker.most_profitable(10)
print(result)
print()
%timeit linker.most_profitable(10)

{'Аватар': '$2686710708', 'Титаник': '$2064812968', 'Звёздные войны: Пробуждение Силы': '$1826310218', 'Мстители: Война бесконечности': '$1731415039', 'Мир Юрского периода': '$1521537444', 'Форсаж 7': '$1325342457', 'Мстители': '$1300538536', 'Гарри Поттер и Дары смерти: Часть II': '$1217505340', 'Мстители: Эра Альтрона': '$1155018048', 'Чёрная Пантера': '$1149926083'}

4.31 ms ± 83.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Лидируют масштабные франшизы и блокбастеры с сильным фанатским базисом — **«Аватар»**, **«Титаник»**, **«Звёздные войны»**, **«Мстители»**. Это говорит о том, что проекты с узнаваемым брендом и продолжением истории привлекают максимально широкую аудиторию.
Ключевыми факторами высокого кассового успеха являются не только качество производства, но и мощная маркетинговая кампания и широкое распределение по кинотеатрам по всему миру. Чем больше экранов и дольше фильм в прокате, тем выше сборы.

#### Топ-10 фильмов по хронометражу

In [38]:
result = linker.longest(10)
print(result)
print()
%timeit linker.longest(10)

{'Крестный отец: Трилогия 1901-1980': '583 mins', 'Тесицюй': '551 mins', 'O.J.: Made in America': '467 mins', 'Лучшие из молодых': '374 mins', 'Двадцатый век': '317 mins', 'Двигаясь вперед, иногда я видел краткие проблески красоты': '288 mins', 'Когда наступит конец света': '287 mins', 'Дюна': '137 mins', 'Человек': '263 mins', 'The Greatest Story Ever Told': '260 mins'}

7.21 ms ± 245 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Самые длинные фильмы — это как правило эпические картины или документальные проекты, которые рассказывают масштабные, глубокие истории и требуют большого экранного времени. Например, **«Крестный отец: Трилогия»** (583 минуты) и **«O.J.: Made in America»** (467 минут) — это многосерийные повествования или расширенные версии.

#### Топ-10 фильмов по стоимости минуты

In [39]:
result = linker.top_cost_per_minute(10)
print(result)
print()
%timeit linker.top_cost_per_minute(10)

{'Терминатор 2 - 3D': 2857142.86, 'Рапунцель: Запутанная история': 2600000.0, 'Лига справедливости': 2500000.0, 'Мстители: Война бесконечности': 2154362.42, 'Хороший динозавр': 2150537.63, 'Люди в чёрном 3': 2122641.51, 'Звёздные войны: Последние джедаи': 2085526.32, 'Рождественская история': 2083333.33, 'В поисках Дори': 2061855.67, 'Хан Соло: Звездные войны. Истории': 2037037.04}

6.16 ms ± 68.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Таким образом, стоимость минуты фильма тесно связана с жанром и технологическим уровнем производства: чем интенсивнее и сложнее технически фильм, тем выше затраты на каждую минуту экранного времени, что отражает стратегию крупных студий вкладывать средства именно в визуальную и технологическую составляющую проектов.

##### Если вы не скачали наш датасет, создайте свой для работы всех методов.  *(==BONUS==)*
  
Для этого укажите ***movieID*** которые хотите использовать, и ***признаки*** которые хотите, чтобы в нем присутствовали. Их нужно передать в   
Поддерживаемые признаки:  
*"title", "director", "release_date", "runtime", "budget", "cumulative_gross", "production_companies", "countries_of_origin", "languages"*  
  
Так же для работы всех методов, вам нужны все признаки!

In [51]:



result = linker.create_dataset(
                    [1, 2, 3, 7, 4, 14, 13],
                    ["title", "director", "release_date", "runtime", "budget", "cumulative_gross", "production_companies", "countries_of_origin", "languages", "other"]
                )
# linker.dict_from_data() # Если хотите использовать свой датасет - раскоментируйте
# linker.dataset_path = ... # Если хотите хардово указать путь к датасету вам сюда

print(result)
print()
%timeit result

Введите название вашего датасета:


 dataname


14, 13, 7, 4, 3, 2, 1, 
Dataset 'dataname.tsv' created successfull
None

8.5 ns ± 0.0709 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)
